# 🎭 Eigenfaces Face Recognition - Predicción

**Usa el modelo pre-entrenado para clasificar fotos nuevas.**

⚠️ **Este notebook NO entrena nada. Solo predice usando el modelo guardado en GitHub.**

**Precisión: 93.93% | Reconoce: 14 personas**

## PASO 1: Autorizar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado")

## PASO 2: Instalar Dependencias

In [ ]:
!pip install mtcnn opencv-python scikit-learn joblib pillow -q
print("✅ Dependencias instaladas")

## PASO 3: Importar Librerías

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import joblib
import matplotlib.pyplot as plt
from mtcnn import MTCNN
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## PASO 4: Descargar y Cargar Modelo desde GitHub

⚠️ **SOLO EDITA ESTO:**
```python
github_usuario = "[REEMPLAZA CON TU USUARIO DE GITHUB]"
```

**Ejemplo:** Si tu usuario es `juanperez`:
```python
github_usuario = "juanperez"
```

In [ ]:
import urllib.request
import tempfile

# 🔧 EDITA SOLO ESTO:
github_usuario = "[REEMPLAZA_CON_TU_USUARIO]"

github_repo = "eigenfaces-recognition"

url_modelo = f"https://raw.githubusercontent.com/{github_usuario}/{github_repo}/main/modelos/modelo_eigenfaces.pkl"
url_nombres = f"https://raw.githubusercontent.com/{github_usuario}/{github_repo}/main/modelos/nombres_unicos.pkl"

print(f"📥 Descargando modelo desde GitHub...")
print(f"   Usuario: {github_usuario}")
print(f"   Repo: {github_repo}\n")

try:
    # Descargar a archivos temporales
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pkl') as tmp1:
        urllib.request.urlretrieve(url_modelo, tmp1.name)
        modelo = joblib.load(tmp1.name)
        print(f"✅ Modelo descargado")
    
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pkl') as tmp2:
        urllib.request.urlretrieve(url_nombres, tmp2.name)
        nombres_unicos = joblib.load(tmp2.name)
        print(f"✅ Lista de personas descargada")
    
    print(f"\n✅ MODELO CARGADO EXITOSAMENTE")
    print(f"\n👥 Personas que reconoce: {len(nombres_unicos)}")
    for i, nombre in enumerate(nombres_unicos, 1):
        print(f"   {i:2d}. {nombre}")
    
    # Inicializar detector de caras
    detector = MTCNN()
    img_size = 64
    print("\n✅ LISTO PARA CLASIFICAR")
    modelo_listo = True
    
except Exception as e:
    print(f"\n❌ ERROR al descargar el modelo")
    print(f"\n   Posibles causas:")
    print(f"   1. El usuario de GitHub no está correcto")
   print(f"   2. El repositorio no existe")
    print(f"   3. Los archivos .pkl no están en modelos/")
    print(f"\n   Error: {e}")
    modelo_listo = False

## PASO 5: Crear Carpeta para Fotos

In [ ]:
if modelo_listo:
    ruta_fotos = '/content/drive/MyDrive/Fotos_para_clasificar'
    os.makedirs(ruta_fotos, exist_ok=True)
    
    print(f"📁 Carpeta de fotos: {ruta_fotos}")
    print(f"\n✅ Sube tus fotos a esta carpeta en Google Drive")
    print(f"   Formatos: JPG, PNG, HEIC")

## PASO 6: Función de Predicción

In [ ]:
if modelo_listo:
    def identificar_cara(ruta_foto, mostrar=True):
        """
        Identifica quién es en una foto.
        
        Args:
            ruta_foto: ruta a la imagen
            mostrar: si True, visualiza la cara detectada
        
        Returns:
            dict con predicción y confianza
        """
        try:
            # Cargar imagen
            img_bgr = cv2.imread(ruta_foto)
            if img_bgr is None:
                return {'error': 'No se pudo cargar la imagen'}
            
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            
            # Redimensionar si es muy grande
            max_dim = 1200
            if img_rgb.shape[0] > max_dim or img_rgb.shape[1] > max_dim:
                ratio = max_dim / max(img_rgb.shape[:2])
                new_size = (int(img_rgb.shape[1] * ratio), int(img_rgb.shape[0] * ratio))
                img_rgb = cv2.resize(img_rgb, new_size)
            
            # Detectar cara
            res = detector.detect_faces(img_rgb)
            if not res:
                return {'error': 'No se detectó cara en la imagen'}
            
            # Usar detección con mejor confianza
            det = max(res, key=lambda x: x['confidence'])
            x, y, w, h = det['box']
            
            # Expandir ROI
            padding = 0.2
            x = max(0, int(x - w * padding))
            y = max(0, int(y - h * padding))
            w_exp = int(w * (1 + 2 * padding))
            h_exp = int(h * (1 + 2 * padding))
            
            x_fin = min(x + w_exp, img_rgb.shape[1])
            y_fin = min(y + h_exp, img_rgb.shape[0])
            
            face = img_rgb[y:y_fin, x:x_fin]
            
            # Procesar cara
            face_gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY)
            face_resized = cv2.resize(face_gray, (img_size, img_size))
            face_resized = cv2.equalizeHist(face_resized)
            
            # Normalizar
            face_vector = face_resized.flatten() / 255.0
            
            # Predicción
            nombre = modelo.predict(face_vector.reshape(1, -1))[0]
            probabilidades = modelo.predict_proba(face_vector.reshape(1, -1))[0]
            confianza = np.max(probabilidades)
            
            # Visualizar
            if mostrar:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
                
                # Cara detectada
                ax1.imshow(face_gray, cmap='gray')
                ax1.set_title(f"Identificado: {nombre.upper()}\nConfianza: {confianza:.1%}",
                             fontsize=14, fontweight='bold')
                ax1.axis('off')
                
                # Top 5 predicciones
                probs_sorted = sorted(zip(nombres_unicos, probabilidades),
                                     key=lambda x: x[1], reverse=True)[:5]
                nombres_top = [p[0] for p in probs_sorted]
                vals_top = [p[1] for p in probs_sorted]
                
                colors = ['#2ecc71' if n == nombre else '#95a5a6' for n in nombres_top]
                
                ax2.barh(nombres_top, vals_top, color=colors)
                ax2.set_xlabel('Probabilidad')
                ax2.set_title('Top 5 Predicciones', fontsize=12, fontweight='bold')
                ax2.set_xlim([0, 1])
                
                for i, v in enumerate(vals_top):
                    ax2.text(v + 0.02, i, f'{v:.1%}', va='center')
                
                plt.tight_layout()
                plt.show()
            
            return {
                'nombre': nombre,
                'confianza': float(confianza),
                'probabilidades': {n: float(p) for n, p in zip(nombres_unicos, probabilidades)}
            }
        
        except Exception as e:
            return {'error': str(e)}
    
    print("✅ Función de predicción lista")

## PASO 7: Clasificar UNA Foto

In [ ]:
if modelo_listo:
    # 🔧 EDITA ESTA RUTA (la foto debe estar en tu Google Drive):
    ruta_foto = '/content/drive/MyDrive/Fotos_para_clasificar/foto1.jpg'
    
    resultado = identificar_cara(ruta_foto, mostrar=True)
    
    if 'error' not in resultado:
        print(f"\n✅ RESULTADO:")
        print(f"   Persona: {resultado['nombre'].upper()}")
        print(f"   Confianza: {resultado['confianza']:.1%}")
    else:
        print(f"❌ Error: {resultado['error']}")

## PASO 8: Clasificar TODA una Carpeta

In [ ]:
if modelo_listo:
    import json
    
    ruta_carpeta = '/content/drive/MyDrive/Fotos_para_clasificar'
    resultados = {}
    
    print(f"🔄 Clasificando fotos...\n")
    
    for foto in sorted(os.listdir(ruta_carpeta)):
        if not foto.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
            continue
        
        ruta_completa = os.path.join(ruta_carpeta, foto)
        print(f"Procesando: {foto:<40}", end=' ', flush=True)
        
        resultado = identificar_cara(ruta_completa, mostrar=False)
        
        if 'error' not in resultado:
            resultados[foto] = {
                'nombre': resultado['nombre'],
                'confianza': resultado['confianza']
            }
            print(f"✅ {resultado['nombre']:<25} ({resultado['confianza']:.1%})")
        else:
            print(f"❌ {resultado['error'][:40]}")
    
    # Mostrar resumen
    print(f"\n📊 RESUMEN:")
    print(f"Total clasificadas: {len(resultados)}")
    
    # Por persona
    personas = {}
    for info in resultados.values():
        nombre = info['nombre']
        personas[nombre] = personas.get(nombre, 0) + 1
    
    print(f"\nPor persona:")
    for nombre, count in sorted(personas.items(), key=lambda x: x[1], reverse=True):
        print(f"   {nombre:<30} {count:>3} fotos")
    
    # Guardar resultados
    ruta_salida = os.path.join(ruta_carpeta, 'resultados.json')
    with open(ruta_salida, 'w') as f:
        json.dump(resultados, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Resultados guardados: {ruta_salida}")

---

## 📋 Resumen

✅ **Listo para usar el modelo pre-entrenado desde GitHub**

1. **PASO 4:** Edita solo tu usuario de GitHub
2. **PASO 7:** Clasifica una foto
3. **PASO 8:** Clasifica una carpeta completa

**Sin entrenar nada. Solo predicción.**